# Trace true provenance of flagged entities\n\n**Author**: Sian Teesdale  \n**Date created**: 2nd July 2026  \n**Dataset Scope**: datasets present in `data/flagged_entities.csv`  \n**Purpose**: For each entity flagged in `1_find_flagged_entities.ipynb` (quality=some, provided by an active local-authority/national-park-authority/development-corporation), trace who **actually** submitted its data — not who it's currently attributed to — and check whether they differ. This directly detects the class of bug in [digital-land/config#2651](https://github.com/digital-land/config/issues/2651) (WOK→WOI, CHW→CHE): an entity's `organisation_entity` is always force-set by the pipeline to match `entity-organisation.csv`'s range assignment (see `digital_land/phase/priority.py`), regardless of who actually supplied the data — so `organisation_entity` alone can't reveal a misattributed range. The real submitter is only visible by following the resource lineage.\n\n**Trace chain** (all via Datasette table filters, batched — no per-entity round trips):\n```\nflagged entity (dataset, entity, assigned_org)\n  -> fact table (per-dataset db):            entity__in=(...)       -> fact hashes\n  -> fact_resource table (per-dataset db):    fact__in=(...)         -> resource hashes\n  -> log table (digital-land db):             resource__in=(...)     -> endpoint\n  -> source table (digital-land db, fetched whole, small): endpoint  -> true_organisation, endpoint_url\n```\n\nValidated against all 4 known-bad entities from the ticket before building this out — every one traces unambiguously to the *correct* organisation (WOI, CHE) rather than the currently-assigned one (WOK, CHW), with `priority=1` on every fact, exactly matching the mechanism above.\n\n**Three false-positive patterns found while validating** — a naive \"true org != assigned org\" diff isn't enough, because these all look identical to a real misattribution bug at the single-entity level:\n- **National seeding** — a central `government-organisation:` (MHCLG, Historic England) legitimately contributes data on behalf of many LPAs (documented in `count_organisations_providers_platform`'s README). Government-organisation contributors are therefore never treated as evidence of misattribution.\n- **Regional aggregation** — a body can also legitimately aggregate on behalf of many *other* `local-authority:`-prefixed orgs, e.g. the Greater London Authority (`local-authority:GLA`) submits `brownfield-site` data for dozens of London boroughs. Detected structurally: a genuine one-to-one range mix-up (WOK→WOI) only ever affects one or two assigned organisations, whereas a regional aggregator's true-org shows up as the source for many different assigned orgs across the dataset. Any true-org serving more than 10 distinct assigned orgs in a dataset is treated as a legitimate aggregator, not a bug.\n- **Local Government Reorganisation succession** — e.g. South Northamptonshire (`SNR`, ended 2021-03-31) merged into West Northamptonshire (`WNUA`, started 2020-04-01); Eden (`EDN`, ended 2023-03-31) merged into Westmorland and Furness (`WFUA`, started 2023-04-01). Historical data legitimately submitted by a since-dissolved predecessor, now correctly attributed to its active legal successor — not a mix-up between two currently-active peers. Detected by checking whether the true org has an `end_date` in `organisation.csv`.\n\n**Classification** (comparing `assigned_org` vs traced `true_org`):\n- **`confirmed_misattribution`** — exactly one non-government, currently-active, non-high-fan-out true org found, and it differs from the assigned org. High-confidence, actionable.\n- **`seeded_by_government`** — every true org is a `government-organisation:` — the normal MHCLG/Historic England seeding pattern, not a bug.\n- **`seeded_by_regional_body`** — the true org is a non-government body that serves as source for many other assigned orgs in the same dataset (e.g. GLA) — a legitimate regional aggregator, not a bug.\n- **`succeeded_inactive_org`** — the true org has an `end_date` — a Local Government Reorganisation succession, not a bug.\n- **`consistent`** — the assigned org is itself among the true orgs (shouldn't normally happen for `quality=some`, but sanity-checked).\n- **`ambiguous`** — multiple different non-government true orgs found, none matching the assigned org and none a clear aggregator. Needs manual review.\n- **`no_resource_found`** — couldn't trace a resource for this entity (edge case, e.g. superseded/historic entity).

In [ ]:
import os

import pandas as pd

from helpers import chunk, fetch_filtered_table_csv, fetch_table_csv, parallel_fetch

DATA_DIR = os.path.join("..", "..", "data")

## 1. Load flagged entities and reference tables

In [ ]:
flagged_df = pd.read_csv(os.path.join(DATA_DIR, "flagged_entities.csv"))
flagged_datasets = sorted(flagged_df["dataset"].unique())
print(f"{len(flagged_df)} flagged entities across {len(flagged_datasets)} datasets")

# Small central tables — fetched whole, once, rather than per-lookup.
endpoint_df = fetch_table_csv("endpoint")
source_df = fetch_table_csv("source")[["endpoint", "organisation"]].drop_duplicates()
org_df = fetch_table_csv("organisation")
org_active = (org_df["end_date"].isna()).set_axis(org_df["organisation"]).to_dict()

## 2. Fetch facts for every flagged entity, per dataset

Batched via `entity__in=(...)` (chunks of 300 ids) so this stays a handful of requests per dataset rather than one per entity.

In [ ]:
fact_jobs = []
for dataset, group in flagged_df.groupby("dataset"):
    entity_ids = group["entity"].astype(int).tolist()
    for batch in chunk(entity_ids, 300):
        fact_jobs.append((
            fetch_filtered_table_csv,
            (dataset, "fact"),
            {"columns": ["entity"], "entity__in": ",".join(map(str, batch))},
        ))

print(f"Fetching facts for {len(flagged_df)} entities across {len(fact_jobs)} batched requests...")
fact_parts = parallel_fetch(fact_jobs)
facts_df = pd.concat(fact_parts, ignore_index=True) if fact_parts else pd.DataFrame(columns=["fact", "entity"])
# facts_df has a "dataset" column added so fact_resource lookups (per-dataset db) know where to query.
facts_df = facts_df.merge(flagged_df[["dataset", "entity"]].drop_duplicates(), on="entity", how="left")
print(f"{len(facts_df)} fact rows, {facts_df['fact'].nunique()} distinct facts")

## 3. Resolve facts to resources

`fact_resource`'s primary key is `rowid`, not `fact` — so `fact` must be requested explicitly in `columns`, unlike tables where the filter column happens to be the PK (auto-included).

In [ ]:
fr_jobs = []
for dataset, group in facts_df.groupby("dataset"):
    fact_hashes = group["fact"].unique().tolist()
    for batch in chunk(fact_hashes, 80):  # fact hashes are 64 chars -- larger batches hit HTTP 414
        fr_jobs.append((
            fetch_filtered_table_csv,
            (dataset, "fact_resource"),
            {"columns": ["fact", "resource"], "fact__in": ",".join(batch)},
        ))

print(f"Resolving facts to resources across {len(fr_jobs)} batched requests...")
fr_parts = parallel_fetch(fr_jobs)
fr_df = pd.concat(fr_parts, ignore_index=True) if fr_parts else pd.DataFrame(columns=["fact", "resource"])
print(f"{len(fr_df)} fact_resource rows, {fr_df['resource'].nunique()} distinct resources")

## 4. Resolve resources to endpoints (`log` table, `digital-land` db)

One `resource` can appear in `log` many times (once per fetch attempt) — dedupe to distinct (resource, endpoint) pairs. Resource hashes are global (not per-dataset), so this is one batched set of requests, not looped per dataset.

In [ ]:
resource_hashes = fr_df["resource"].unique().tolist()
log_jobs = [
    (
        fetch_filtered_table_csv,
        ("digital-land", "log"),
        {"columns": ["resource", "endpoint"], "resource__in": ",".join(batch)},
    )
    for batch in chunk(resource_hashes, 80)
]

print(f"Resolving {len(resource_hashes)} resources to endpoints across {len(log_jobs)} batched requests...")
log_parts = parallel_fetch(log_jobs)
log_df = (
    pd.concat(log_parts, ignore_index=True).drop_duplicates(subset=["resource", "endpoint"])
    if log_parts else pd.DataFrame(columns=["resource", "endpoint"])
)
print(f"{len(log_df)} distinct (resource, endpoint) pairs")

## 5. Join the full chain and derive the true organisation per entity

In [ ]:
chain = (
    facts_df.merge(fr_df, on="fact")
    .merge(log_df, on="resource")
    .merge(source_df, on="endpoint")
    .merge(endpoint_df[["endpoint", "endpoint_url"]], on="endpoint")
)
print(f"{len(chain)} chain rows linking entity -> true organisation")

true_org_per_entity = (
    chain.groupby("entity")["organisation"]
    .apply(lambda s: sorted(set(s)))
    .reset_index(name="true_orgs")
)
true_endpoint_per_entity = (
    chain.groupby("entity")["endpoint_url"]
    .apply(lambda s: sorted(set(s)))
    .reset_index(name="true_endpoint_urls")
)

result = flagged_df.merge(true_org_per_entity, on="entity", how="left")
result = result.merge(true_endpoint_per_entity, on="entity", how="left")

## 6. Classify

In [ ]:
GOV_PREFIX = "government-organisation:"
FANOUT_THRESHOLD = 10  # true-orgs serving more than this many distinct assigned orgs are treated as legitimate aggregators (e.g. GLA), not bugs


def classify_stage1(row):
    true_orgs = row["true_orgs"]
    assigned = row["organisation"]
    if not isinstance(true_orgs, list) or len(true_orgs) == 0:
        return "no_resource_found"
    if assigned in true_orgs:
        return "consistent"
    non_gov = [o for o in true_orgs if not o.startswith(GOV_PREFIX)]
    if len(non_gov) == 0:
        return "seeded_by_government"
    if len(non_gov) == 1:
        return "confirmed_misattribution"
    return "ambiguous"


result["classification"] = result.apply(classify_stage1, axis=1)
result["true_org"] = result["true_orgs"].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else None)

# Stage 2: reclassify high-fan-out true-orgs (regional aggregators like GLA) out of confirmed_misattribution.
fanout = (
    result[result["classification"] == "confirmed_misattribution"]
    .groupby(["dataset", "true_org"])["organisation"]
    .nunique()
)
high_fanout_pairs = set(fanout[fanout > FANOUT_THRESHOLD].index)

result.loc[
    result.apply(lambda r: (r["dataset"], r["true_org"]) in high_fanout_pairs, axis=1)
    & (result["classification"] == "confirmed_misattribution"),
    "classification",
] = "seeded_by_regional_body"

# Stage 3: reclassify cases where the true_org is itself inactive (has an end_date) --
# this is a Local Government Reorganisation succession (e.g. SNR -> WNUA, EDN -> WFUA),
# the assigned (active) org legitimately inheriting historical data from its dissolved
# predecessor, not a genuine range mix-up between two currently-active peers.
result["true_org_active"] = result["true_org"].map(org_active)
result.loc[
    (result["classification"] == "confirmed_misattribution") & (result["true_org_active"] == False),
    "classification",
] = "succeeded_inactive_org"

result["classification"].value_counts()

## 7. Explore confirmed misattributions

These are the actionable ones — high confidence, same fingerprint as the known #2651 examples.

In [ ]:
confirmed = result[result["classification"] == "confirmed_misattribution"]
by_org_pair = (
    confirmed.groupby(["dataset", "organisation", "true_org"])
    .agg(entities=("entity", "nunique"))
    .sort_values("entities", ascending=False)
)
by_org_pair

In [ ]:
by_org_pair['entities'].sum()

In [ ]:
confirmed[["dataset", "entity", "organisation", "true_org", "entity_url", "true_endpoint_urls"]].tail(30)

## 8. Export prioritized list

`confirmed_misattribution` sorted to the top — this is the list to take back to the ticket.

In [ ]:
priority_order = {
    "confirmed_misattribution": 0,
    "ambiguous": 1,
    "no_resource_found": 2,
    "succeeded_inactive_org": 3,
    "seeded_by_regional_body": 4,
    "seeded_by_government": 5,
    "consistent": 6,
}
result["_priority"] = result["classification"].map(priority_order)
result = result.sort_values(["_priority", "dataset", "organisation", "entity"]).drop(columns="_priority")

export_cols = [
    "dataset", "entity", "name", "reference", "organisation", "organisation_name",
    "quality", "entity_url", "classification", "true_org", "true_orgs", "true_endpoint_urls",
]
out_path = os.path.join(DATA_DIR, "flagged_entities_with_provenance.csv")
result[export_cols].to_csv(out_path, index=False)
print(f"Saved {len(result)} rows to {out_path}")